## An Example: NiBabel Image Classes

In the previous chapters we saw that classes are like blueprints for objects. It frequently happens that many of these blueprints share similarities.
Take, for example, this 'taxonomic tree' of some of the image formats that NiBabel can read:

```
SpatialImage
├── AnalyzeImage (basic Mayo Analyze format)
│   ├── Spm99AnalyzeImage (SPM2 variant of the Analyze format)
│   │   └── Spm2AnalyzeImage (SPM99 variant of the Analyze format)
│   └── Nifti1Image (NIfTI1 format)
│       └── Nifti2Image (NIfTI2 format)
├── Minc1Image (MINC1 format)
│   └── Minc2Image (MINC2 format)
├── EcatImage (ECAT format)
├── MGHImage (FreeSurfer MGH format)
└── PARRECImage (PAR/REC format)
```

We've been using the `Nifti1Image` format in our assignments, which has pixel data, dimensions, an affine matrix (pixel spacing, rotation, translation), and more metadata. You can probably imagine that this affine matrix is not something unique to the Nifti image format. And you can probably also imagine that the NIfTI2 format must have similarities to the NIfTI1 format. As a matter of fact, the Nifti, SPM, and Analyze image formats are all similar, because Nifti and SPM were both historically derived from the older Analyze format.

In order to prevent code duplication, and to keep the API consistent, classes can also *inherit* (or *derive*) from each other. In the hierarchical tree above, you can see that the NIfTI2 format inherits functionality from the NIfTI1 format, which, in its turn, inherits functionality from the basic Mayo Analyze format.

All these image classes ultimately inherit from the `SpatialImage` class, which is the *base class* for all other image classes. `SpatialImage` is not a file format on its own, but just a class providing a generic blueprint for all image classes: it contains class attributes and class methods that are common to all derived classes, such as `SpatialImage.affine`, and `SpatialImage.shape`.

The Python function `isinstance` can be used to check if an object belongs to a certain class. Note how a `Nifti1Image` *also* belongs to its *parent* classes:

In [29]:
import nibabel
import numpy as np
from nibabel.spatialimages import SpatialImage


# Create a dummy NIfTI1 image.
data = np.zeros([3, 3, 3])
affine = np.eye(4)
img = nibabel.Nifti1Image(data, affine)

# Iterate over some image classes and check if img belongs to that class.
for parent_class in [nibabel.Nifti2Image, nibabel.Nifti1Image, nibabel.AnalyzeImage, SpatialImage]:
    belongs = 'belongs' if isinstance(img, parent_class) else 'does NOT belong'
    print(f'A {type(img).__name__} image {belongs} to the {parent_class.__name__} class.')

A Nifti1Image image does NOT belong to the Nifti2Image class.
A Nifti1Image image belongs to the Nifti1Image class.
A Nifti1Image image belongs to the AnalyzeImage class.
A Nifti1Image image belongs to the SpatialImage class.


Note that this means that a `Nifti1Image` can also be used as an `AnalyzeImage` or as a `SpatialImage`, because it has derived all its properties and functionality! If you'd write a function that works for a `SpatialImage`, it would work for *all* classes deriving from it, because they all share the same API.

The base class `SpatialImage` also declares a class method `load()`. However, calling it raises a `NotImplementedError` because the `SpatialImage` class does not represent a file format; it is merely a base class. The `load()` method, however, acts as a blueprint method for derived classes. The derived classes should *override* this method with their own implementation for loading NIfTI1/2, Analyze, or SPM2/99 images.

## Another Example: PyTorch Modules

Another example of a base class and derived classes are the building blocks in [PyTorch](https://docs.pytorch.org/) neural networks. When you've used PyTorch, you probably have seen or used convolution layers (e.g. `nn.Conv2d`,`nn.Conv3d`), activation layers (e.g. `nn.ReLu`, `nn.Sigmoid`), dropout layers, loss functions, etc. All these building blocks can be assembled into a pipeline or graph (like in the figure below), because they all share the same interface. This uniform interface is provided by their base class: `nn.Module`.

```
    Input
      │
      ▼
┌───────────┐    ┌─────────┐    ┌────────────┐    ┌────────────┐
│ nn.Linear │--->│ nn.ReLU │--->│ nn.Dropout │--->│ nn.Linear  │
│ forward() │    │forward()│    │ forward()  │    │ forward()  │
└───────────┘    └─────────┘    └────────────┘    └────────────┘
                                                         │
                                              Prediction │
                                                         ▼
                                                  ┌────────────┐
                                       Target --->│ nn.MSELoss │
                                                  │ forward()  │
                                                  └────────────┘
                                                         │
                                                         ▼
                                                    Loss value
```

 All building blocks in the figure above are instances of the class `nn.Module`. This base class provides class methods such as `forward()`, `train()`, and `zero_grad()`. The method `forward()` is automatically called by the graph to perform the computation, and should be overridden in derived classes; the implementation of `forward()` defines the computation performed by a PyTorch module, distinguishing the behavior of a convolution layer, an activation layer, and a loss function.

Similar to the `load()` method of the `SpatialImage` class in NiBabel, the `forward()` method of the `nn.Module` base class raises a `NotImplementedError` when you call it, because this class does not represent the implementation of a module; it is merely a base class providing the generic blueprint for a module.

## Inheriting From a Base Class

Let's investigate how inheriting from the PyTorch `nn.Module` base class works by implementing a very simplified version ourselves. The base class could look like this:

In [55]:
class Module:
    def __init__(self):
        print("Initializing an instance of the class", type(self).__name__)

    def forward(self, x):
        """
        Executes the forward pass of the module.
        Should be overridden by all derived classes.

        :param x: The input to the module.
        :return: The result of the computation.
        """
        raise NotImplementedError

    # This dunder method allows an object to be called like a function.
    def __call__(self, x):
        return self.forward(x)


# Create an instance of the base class 'Module' and try to call it.
module = Module()
module(1)  # This will call the __call__() class method.

Initializing an instance of the class Module


NotImplementedError: 

As you can see in the traceback, calling `module(1)` calls the `__call__()` dunder method of the `Module` class, which in turn calls `forward()`. Because this is a base class, we defined the `forward()` method to raise a `NotImplementedError`.

The derived classes, that inherit from `Module`, should override this `forward()` method. Note that they don't necessarily have to override the methods that *are* implemented in the base class, like `__init__()` and `__call__()`.

Now let's create two classes that derive from `Module`. The class to derive from can be given as an argument to the new class, like this:

In [68]:
class Double(Module):  # Derive from 'Module'
    def forward(self, x):
        print(f"Double {x}")
        return x * 2


class AddOne(Module):  # Derive from 'Module'
    def forward(self, x):
        print(f"Add 1 to {x}")
        return x + 1

Note how the `__init__()` method of the base class is called when we initialize an object of the derived classes:

In [67]:
double = Double()
add_one = AddOne()

Initializing an instance of the class Double
Initializing an instance of the class AddOne


And note how the `__call__()` method of the base class is called, which calls the -now implemented- forward() method:

In [71]:
print(double(3))
print(add_one(1))

Double 3
6
Add 1 to 1
2


Both the objects `double` and `add_one` are instances of the (base) class `Module`:

In [92]:
isinstance(add_one, Module) and isinstance(double, Module)

True

In the same fashion, we can also implement a class for executing sequence of modules. This class should have a class for storing a list of modules, and it's `forward()` method should execute all these modules in sequence, passing the output of a previous module as input to the next.

In order to store the list of modules, we need to override the `__init__` method of the base class. In the new `__init__` method, we can call `super().__init__()` to *also* execute the `__init__()` function of the parent class. The `super()` function retrieves the parent class (in this case `Module`):

In [72]:
class Sequential(Module):  # Derive from 'Module'
    def __init__(self, modules):
        super().__init__()  # Call the parent class __init__()
        self._modules = modules  # Store the given list of modules.

    def forward(self, x):
        # Iterate over all modules and execute them, passing the output
        # of a previous module as input to the next.
        for module in self._modules:
            x = module(x)
        return x

Now let's use our new `Sequential` module to build our own miniature model with a sequence of 3 operations:

In [76]:
model = Sequential([Double(), AddOne(), Double()])

Initializing an instance of the class Double
Initializing an instance of the class AddOne
Initializing an instance of the class Double
Initializing an instance of the class Sequential


This is what happens if we execute the model with the value `5` as input:

In [89]:
model(5)  # (5 × 2 + 1) × 2 = 22

Double 5
Add 1 to 10
Double 11


22

And this is what happens if we execute the model with a NumPy array as input:

In [88]:
x = np.array([3, 5, 7, 11])
y = model(x)
print(y)

Double [ 3  5  7 11]
Add 1 to [ 6 10 14 22]
Double [ 7 11 15 23]
[14 22 30 46]


## Try It Out Yourself

> **External course material**
>
> Review these pages and do the exercises:
>
> * <https://www.w3schools.com/python/python_inheritance.asp>
> * <https://www.w3schools.com/python/python_polymorphism.asp>